In [ ]:
reset

In [ ]:
import os
import sys
# add path to custom functions
module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path+"/py_functions")
# import custom functions
from map_plot_tools import *
from line_plot_tools import *
from colorbar_funcs import *
from data_funcs import *

import xarray as xr
xr.set_options(keep_attrs=True)
import numpy as np
import metpy.calc as mp

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
from shapely.geometry.polygon import LinearRing

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
from matplotlib.colors import TwoSlopeNorm
from matplotlib import cm
from matplotlib.colors import ListedColormap,LinearSegmentedColormap
import cmocean.cm as cmo
import seaborn as sns
# settings
%config InlineBackend.figure_format = 'retina'

# top level data directory
dpath0='/Users/dervlamk/OneDrive/research/cesm/lig_icesm1.2'
# save figs here
opath='/Users/dervlamk/OneDrive/research/cesm/lig_icesm1.2/figs'

In [ ]:
# File Paths

files = {}
for sim in ['pi']:
    run = 'PI'
    files[sim] = {}
    for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS']:
        files[sim][varn] = f'{dpath0}/{run}/dh.precIsotopes.atm.iPI.nc'
    for varn in ['PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS']:
        files[sim][varn] = f'{dpath0}/{run}/o.precIsotopes.atm.iPI.nc'
    for varn in ['PRECC', 'PRECL']:
        files[sim][varn] = f'{dpath0}/{run}/atm.2d.vars.iPI.climo.nc'

for sim in ['lig']:
    run='LIG'
    files[sim] = {}
    for varn in ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS']:
        files[sim][varn] = f'{dpath0}/{run}/dh.precIsotopes.atm.iLIG127K.nc'
    for varn in ['PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS']:
        files[sim][varn] = f'{dpath0}/{run}/o.precIsotopes.atm.iLIG127K.nc'
    for varn in ['PRECC', 'PRECL']:
        files[sim][varn] = f'{dpath0}/{run}/atm.2d.vars.iLIG127K.climo.nc'


In [ ]:
# Load Data

dat={}
sims=['pi','lig']
varns = ['PRECRC_H2Or', 'PRECRL_H2OR', 'PRECSC_H2Os', 'PRECSL_H2OS', 'PRECRC_HDOr', 'PRECRL_HDOR', 'PRECSC_HDOs', 'PRECSL_HDOS', 
         'PRECRC_H216Or', 'PRECRL_H216OR', 'PRECSC_H216Os', 'PRECSL_H216OS', 'PRECRC_H218Or', 'PRECRL_H218OR', 'PRECSC_H218Os', 'PRECSL_H218OS', 
         'PRECC', 'PRECL']

for sim in sims:
    dat[sim]={}
    for varn in varns:
        dat[sim][varn]=xr.open_dataset(files[sim][varn])[varn]

In [ ]:
# Calculate Climatologies

dDp={}
d18Op={}
prec={}
sims=['pi','lig']

for sim in sims:
    ptiny=1e-18;
    
    ## Precipitation
    # calculate total precip from convective and large-scale prec vars (snow+rain). convert from m/s to mm/day
    prec[sim] = (dat[sim]['PRECC'] + dat[sim]['PRECL'])*1000*60*60*24
    # calculate precipitation weights by month
    if sim in ['pi']:
        annual_total_p = prec[sim].sum(dim="time")
    if sim in ['lig']:
        annual_total_p = prec[sim].sum(dim="month")
    pWeights = prec[sim]/annual_total_p
    
    ## Hydrogen
    ph = dat[sim]['PRECRC_H2Or'] + dat[sim]['PRECRL_H2OR'] + dat[sim]['PRECSC_H2Os'] + dat[sim]['PRECSL_H2OS']
    pd = dat[sim]['PRECRC_HDOr'] + dat[sim]['PRECRL_HDOR'] + dat[sim]['PRECSC_HDOs'] + dat[sim]['PRECSL_HDOS']
    # replace very small ph values with a tiny value
    ph = ph.where(ph > ptiny, ptiny) 
    # turn into per mil notation
    dd = (pd/ph - 1)*1000 
    # Multiply isotope values by weights
    dDp[sim] = dd*pWeights
    
    ## Oxygen
    p16o = dat[sim]['PRECRC_H216Or'] + dat[sim]['PRECRL_H216OR'] + dat[sim]['PRECSC_H216Os'] + dat[sim]['PRECSL_H216OS']
    p18o = dat[sim]['PRECRC_H218Or'] + dat[sim]['PRECRL_H218OR'] + dat[sim]['PRECSC_H218Os'] + dat[sim]['PRECSL_H218OS']
    # replace very small ph values with a tiny value
    p16o = p16o.where(p16o > ptiny, ptiny)
    # turn into per mil notation
    do = (p18o/p16o - 1)*1000 
    # Multiply isotope values by weights
    d18Op[sim] = do*pWeights
    

## Precip

### Precip PI Climo

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
font_kw={'color':'k', 'weight':'bold', 'size':16, 'horizontalalignment':'center'}
lw=1

# colormap specs
cmap2=plt.colormaps['Blues']
vmin2=0
vmax2=12
levels=np.linspace(vmin2, vmax2, 13)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)

# map specs
lat = prec['pi'].lat
lon = prec['pi'].lon
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]

# core info
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff=[12.42523674, -0.67940596]


fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(15,15), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1.025,'iCESM1.2 PI Precip Climo', **font_kw)

rows = [0,0,0,1,1,1,2,2,2,3,3,3]
cols = [0,1,2,0,1,2,0,1,2,0,1,2]
mons = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
idx = ['0','1','2','3','4','5','6','7','8','9','10','11']

for i in range(0,12):
    rown = rows[i]
    coln = cols[i]
    ax[rown,coln].text(((map_bnds[0]+map_bnds[1])/2), map_bnds[3]+0.5, mons[i]+ '[' +idx[i]+ ']', **font_kw)
    cf=ax[rown,coln].pcolormesh(lon, lat, prec['pi'].isel(time=i),
                                cmap=cmap2, norm=norm2, transform=trans)

for i,ax in enumerate(ax.flat):
    # add IMERG climatology contours
    cs=ax.contour(P_OBS.lon, P_OBS.lat, P_OBS[i], levels=np.linspace(5,16,10), linewidths=0.8, colors='fuchsia', transform=trans)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES, linewidth=0.5)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)

cbar_ax = fig.add_axes([0.1, -0.05, 0.8, 0.025])
cbar = fig.colorbar(cf, orientation='horizontal', extend='max', cax=cbar_ax)
cbar.set_label('[mm day$^{-1}$]', labelpad=5, size=14, rotation=0)
cbar.ax.tick_params(labelsize=14)

### Precip LIG Climo

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
font_kw={'color':'k', 'weight':'bold', 'size':16, 'horizontalalignment':'center'}
lw=1

# colormap specs
cmap2=plt.colormaps['Blues']
vmin2=0
vmax2=12
levels=np.linspace(vmin2, vmax2, 13)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)

# map specs
lat = prec['pi'].lat
lon = prec['pi'].lon
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]

# core info
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff=[12.42523674, -0.67940596]


fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(15,15), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1.025,'iCESM1.2 LIG Precip Climo', **font_kw)

rows = [0,0,0,1,1,1,2,2,2,3,3,3]
cols = [0,1,2,0,1,2,0,1,2,0,1,2]
mons = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
idx = ['0','1','2','3','4','5','6','7','8','9','10','11']

for i in range(0,12):
    rown = rows[i]
    coln = cols[i]
    ax[rown,coln].text(((map_bnds[0]+map_bnds[1])/2), map_bnds[3]+0.5, mons[i]+ '[' +idx[i]+ ']', **font_kw)
    cf=ax[rown,coln].pcolormesh(lon, lat, prec['lig'].isel(month=i),
                                cmap=cmap2, norm=norm2, transform=trans)

for i,ax in enumerate(ax.flat):
    # add IMERG climatology contours
    cs=ax.contour(P_OBS.lon, P_OBS.lat, P_OBS[i], levels=np.linspace(5,16,10), linewidths=0.8, colors='fuchsia', transform=trans)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES, linewidth=0.5)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)

cbar_ax = fig.add_axes([0.1, -0.05, 0.8, 0.025])
cbar = fig.colorbar(cf, orientation='horizontal', extend='max', cax=cbar_ax)
cbar.set_label('[mm day$^{-1}$]', labelpad=5, size=14, rotation=0)
cbar.ax.tick_params(labelsize=14)

### Precip Diff

In [ ]:
## Load IMERG observational precip data
filen='/Users/dervlamk/OneDrive/research/obs_data/satellite/imerg/imerg.climo.gn.nc'
P_OBS=xr.open_dataset(f'{filen}').precipitation
P_OBS.attrs['source_id']='IMERG'
# convert lons from -180:180 to 0:360
nroll=int(len(P_OBS.lon)/2)
P_OBS=P_OBS.roll(lon=nroll)
nx=len(P_OBS.lon)
P_OBS['lon']=np.linspace(0,360,nx)
P_OBS=P_OBS.interp_like(prec['pi'],method='linear')

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
font_kw={'color':'k', 'weight':'bold', 'size':16, 'horizontalalignment':'center'}
lw=1

# colormap specs
pcmap,_,_,_=get_settings(field='precip', diff=True)
pvmin=-3
pvmax=3
plevels=np.linspace(pvmin, pvmax, 25)
pnorm=mpl.colors.BoundaryNorm(plevels, pcmap.N)

# map specs
lat = prec['pi'].lat
lon = prec['pi'].lon
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-130., -65., -10., 42.]

# core info
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff=[12.42523674, -0.67940596]


fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(15,15), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1.025,'iCESM1.2 LIG$-$PI $\Delta$PRECIPITATION', **font_kw)

rows = [0,0,0,1,1,1,2,2,2,3,3,3]
cols = [0,1,2,0,1,2,0,1,2,0,1,2]
mons = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
idx = ['0','1','2','3','4','5','6','7','8','9','10','11']

for i in range(0,12):
    rown = rows[i]
    coln = cols[i]
    ax[rown,coln].text(((map_bnds[0]+map_bnds[1])/2), map_bnds[3]+0.5, mons[i]+ '[' +idx[i]+ ']', **font_kw)
    cf=ax[rown,coln].pcolormesh(lon, lat, (prec['lig'].isel(month=i)-prec['pi'].isel(time=i)),
                                cmap=pcmap, norm=pnorm, transform=trans)

for i,ax in enumerate(ax.flat):
    # add IMERG climatology contours
    cs=ax.contour(P_OBS.lon, P_OBS.lat, P_OBS[i], levels=np.linspace(5,16,10), linewidths=0.8, colors='fuchsia', transform=trans)
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES, linewidth=0.5)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)

cbar_ax = fig.add_axes([0.1, -0.05, 0.8, 0.025])
cbar = fig.colorbar(cf, orientation='horizontal', extend='both', cax=cbar_ax)
cbar.set_label('[mm day$^{-1}$]', labelpad=5, size=14, rotation=0)
cbar.ax.tick_params(labelsize=14)

### Annual Cycle

In [ ]:
lat_min = 18
lat_max = 33
lon_min = 247
lon_max = 255

pobs_clip=P_OBS.sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).mean(dim=['lat','lon'])
ppi_clip=prec['pi'].sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).mean(dim=['lat','lon'])
plig_clip=prec['lig'].sel(lon=slice(lon_min,lon_max), lat=slice(lat_min,lat_max)).mean(dim=['lat','lon'])

In [ ]:
labels=['IMERG','PI','LIG']
tkw = {'axis': 'both', 'direction':'in', 'labelsize': 'x-large'} 
text_kw={'size': 'xx-large', 'weight': 'bold',  'color': 'k', 'ha':'center','va':'bottom'}
legend_prop={'size':'x-large', 'weight':'bold'}
legend_kw={'labelcolor':'linecolor', 'ncols':1, 'frameon':False}
idx=[0,1,2,3,4,5,6,7,8,9,10,11]

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(9,6), layout='constrained')
ax.text(5.5, 6.15, 'MEAN RAINFALL [113°W$-$105°W]', rotation=0, **text_kw)
 
ax.plot(idx, pobs_clip, c='k', ls='-', lw=2, label='IMERG')
ax.plot(idx, ppi_clip, c='peru', ls='-', lw=2, label='PI')
ax.plot(idx, plig_clip, c='#9a0200', ls='-', lw=2, label='LIG')

ax.set(xlim=[0, 11], ylim=[0,6.1])
ax.set_xlabel('MONTH', weight='bold', size='x-large')
ax.set_xticks([0,1,2,3,4,5,6,7,8,9,10,11])
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_yticks([1,2,3,4,5,6])
ax.set_ylabel('[mm/day]', weight='bold', size='x-large')
ax.tick_params(**tkw)
    
ax.legend(loc=2, prop=legend_prop, **legend_kw)

## Isotopes

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
font_kw={'color':'k', 'weight':'bold', 'size':16, 'horizontalalignment':'center'}
lw=1

# colormap specs
cmap2=plt.colormaps['BrBG']
vmin2=-3
vmax2=3
levels=np.linspace(vmin2, vmax2, 13)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)

# map specs
lat = prec['pi'].lat
lon = prec['pi'].lon
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-130., -65., -10., 42.]

# core info
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff=[12.42523674, -0.67940596]


fig, ax = plt.subplots(nrows=4, ncols=3, figsize=(15,15), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1.025,'iCESM1.2 LIG$-$PI Precip', **font_kw)

rows = [0,0,0,1,1,1,2,2,2,3,3,3]
cols = [0,1,2,0,1,2,0,1,2,0,1,2]
mons = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
idx = ['0','1','2','3','4','5','6','7','8','9','10','11']

for i in range(0,12):
    rown = rows[i]
    coln = cols[i]
    ax[rown,coln].text(((map_bnds[0]+map_bnds[1])/2), map_bnds[3]+0.5, mons[i]+ '[' +idx[i]+ ']', **font_kw)
    cf=ax[rown,coln].pcolormesh(lon, lat, (dDp['lig'].isel(month=i)-dDp['pi'].isel(time=i)),
                                cmap=cmap2, vmin=vmin2, vmax=vmax2, transform=trans)

for i,ax in enumerate(ax.flat):
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES, linewidth=0.5)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)
   

## Old

### PI

In [ ]:
## Hydrogen Isotopes

# add up 1H and 2H
ph=dh_pi.PRECRC_H2Or + dh_pi.PRECRL_H2OR + dh_pi.PRECSC_H2Os + dh_pi.PRECSL_H2OS
pd=dh_pi.PRECRC_HDOr + dh_pi.PRECRL_HDOR + dh_pi.PRECSC_HDOs + dh_pi.PRECSL_HDOS

# replace very small ph values with a tiny value
ptiny=1e-18;
ph=ph.where(ph > ptiny, ptiny)

# turn into per mil notation
dDp_pi=(pd/ph - 1)*1000

In [ ]:
## Oxygen Isotopes

# add up 16s and 18s
p16o=o_pi.PRECRC_H216Or + o_pi.PRECRL_H216OR + o_pi.PRECSC_H216Os + o_pi.PRECSL_H216OS
p18o=o_pi.PRECRC_H218Or + o_pi.PRECRL_H218OR + o_pi.PRECSC_H218Os + o_pi.PRECSL_H218OS

# replace very small p16o values with a tiny value
ptiny=1e-18;
p16o=p16o.where(p16o > ptiny, ptiny)

# turn into per mil notation
d18Op_pi=(p18o/p16o - 1)*1000

In [ ]:
## Precipitation

# calculate total precip from convective and large-scale prec vars (snow+rain). units are m/s
prec_pi=cam_pi.PRECC + cam_pi.PRECL

# convert to mm/day
prec_pi=prec_pi*1000*60*60*24

In [ ]:
## Calculate Avg d18O and dD weighted by precipitation

# Total annual precip
prec_pi_total = prec_pi.sum(dim="time")

# Weight by month
pWeights_pi = prec_pi/prec_pi_total

# Multiply isotope values by weights
dOweightedvalues_pi = d18Op_pi*pWeights_pi
dDweightedvalues_pi = dDp_pi*pWeights_pi

# Calculate Weighted Avg
d18Op_pi_avg = dOweightedvalues_pi.sum(dim="time")
dDp_pi_avg = dDweightedvalues_pi.sum(dim="time")

In [ ]:
for key in ['pi']:
    dD[key] = dDp_pi_avg
    d18O[key] = d18Op_pi_avg
    prec[key] = prec_pi

In [ ]:
dOweightedvalues_pi

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# var specs
im=5 #start month
em=9 #end month
# mean annual = 0:12
# JAS = 6:9
# JJAS = 5:9
# ** for DJF, replace 'im:em' with '[11,0,1]'
# plot specs
lw=1
font={'color':  'k', 'weight': 'bold', 'size': 14, 'horizontalalignment': 'left'}
titles=np.array(['$\delta$D$_{precip}$', 'Precipitation'])
# colormap
cmap1=cm.cubehelix #cmo.thermal
vmin1=-15
vmax1=0
levels=np.linspace(vmin1, vmax1, 31)
norm1=mpl.colors.BoundaryNorm(levels, cmap1.N)
cmap2=cm.Blues
vmin2=0
vmax2=15
levels=np.linspace(vmin2, vmax2, 31)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-130., -65., -10., 45.]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20,10), subplot_kw={'projection': proj})
fig.text(.5,.85,'iCESM1.2 PI (1850) June-July-August-September Climatologies', size=16, weight='bold', ha='center')

ax[0].pcolormesh(lon, lat, dDweightedvalues_pi[im:em,:,:].mean(dim="time"), cmap=cmap1, vmin=vmin1, vmax=vmax1, transform=trans)
ax[1].pcolormesh(lon, lat, prec_pi[im:em,:,:].mean(dim="time"), cmap=cmap2, vmin=vmin2, vmax=vmax2, transform=trans)

for i in [0,1]:
    ax[i].coastlines()
    #ax[i].add_feature(cfeature.LAND, fc=(0, 0, 0),zorder=2)
    ax[i].set_extent(map_bnds, crs=trans)
    gl=ax[i].gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)
    gl.top_labels=False; gl.right_labels=False


fig.subplots_adjust(bottom=0, top=.95)

cbar_ax1 = fig.add_axes([0.48, 0.175, 0.015, 0.6])
cbar1 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm1, cmap=cmap1), orientation='vertical', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='bold', y=1.05, labelpad=-20, rotation=0)
cbar1.ax.tick_params(labelsize=14)
for tick in cbar1.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')

cbar_ax2 = fig.add_axes([0.905, 0.175, 0.015, 0.6])
cbar2 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm2, cmap=cmap2), orientation='vertical', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='bold', y=1.05, labelpad=-20, rotation=0)
cbar2.ax.tick_params(labelsize=14)
for tick in cbar2.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')

#plt.savefig("figs/ep_precip_bias.pdf")

### LIG

In [ ]:
## Precipitation

# calculate total precip from convective and large-scale prec vars (snow+rain). units are m/s
prec_lig=cam_lig.PRECC + cam_lig.PRECL

# convert to mm/day
prec_lig=prec_lig*1000*60*60*24

In [ ]:
## Hydrogen Isotopes

# add up 1H and 2H
ph=dh_lig.PRECRC_H2Or + dh_lig.PRECRL_H2OR + dh_lig.PRECSC_H2Os + dh_lig.PRECSL_H2OS
pd=dh_lig.PRECRC_HDOr + dh_lig.PRECRL_HDOR + dh_lig.PRECSC_HDOs + dh_lig.PRECSL_HDOS

# replace very small ph values with a tiny value
ptiny=1e-18;
ph=ph.where(ph > ptiny, ptiny)

# turn into per mil notation
dDp_lig=(pd/ph - 1)*1000

In [ ]:
## Oxygen Isotopes

# add up 16s and 18s
p16o=o_lig.PRECRC_H216Or + o_lig.PRECRL_H216OR + o_lig.PRECSC_H216Os + o_lig.PRECSL_H216OS
p18o=o_lig.PRECRC_H218Or + o_lig.PRECRL_H218OR + o_lig.PRECSC_H218Os + o_lig.PRECSL_H218OS

# replace very small p16o values with a tiny value
ptiny=1e-18;
p16o=p16o.where(p16o > ptiny, ptiny)

# turn into per mil notation
d18Op_lig=(p18o/p16o - 1)*1000

In [ ]:
## Calculate Avg d18O and dD weighted by precipitation

# Total annual precip
prec_lig_total = prec_lig.sum(dim="month")

# Weight by month
pWeights_lig = prec_lig/prec_lig_total

# Multiply isotope values by weights
dOweightedvalues_lig = d18Op_lig*pWeights_lig
dDweightedvalues_lig = dDp_lig*pWeights_lig

# Calculate Weighted Avg
d18Op_lig_avg = dOweightedvalues_lig.sum(dim="month")
dDp_lig_avg = dDweightedvalues_lig.sum(dim="month")

In [ ]:
for key in ['lig']:
    dD[key] = dDp_lig_avg
    d18O[key] = d18Op_lig_avg
    prec[key] = prec_lig

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# var specs
im=5 #start month
em=9 #end month
# mean annual = 0:12
# JAS = 6:9
# JJAS = 5:9
# ** for DJF, replace 'im:em' with '[11,0,1]'
# plot specs
lw=1
font={'color':  'k', 'weight': 'bold', 'size': 14, 'horizontalalignment': 'left'}
titles=np.array(['$\delta$D$_{precip}$', 'Precipitation'])
# colormap
cmap1=cm.cubehelix #cmo.thermal
vmin1=-15
vmax1=0
levels=np.linspace(vmin1, vmax1, 31)
norm1=mpl.colors.BoundaryNorm(levels, cmap1.N)
cmap2=cm.Blues
vmin2=0
vmax2=15
levels=np.linspace(vmin2, vmax2, 31)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)
# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-130., -65., -10., 45.]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(20,10), subplot_kw={'projection': proj})
fig.text(.5,.85,'iCESM1.2 LIG (127ka) July-August-September Climatologies', size=16, weight='bold', ha='center')

ax[0].pcolormesh(lon, lat, dDweightedvalues_lig[im:em,:,:].mean(dim="month"), cmap=cmap1, vmin=vmin1, vmax=vmax1, transform=trans)
ax[1].pcolormesh(lon, lat, prec_lig[im:em,:,:].mean(dim="month"), cmap=cmap2, vmin=vmin2, vmax=vmax2, transform=trans)

for i in [0,1]:
    ax[i].coastlines()
    ax[i].set_extent(map_bnds, crs=trans)
    gl=ax[i].gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)
    gl.top_labels=False; gl.right_labels=False


fig.subplots_adjust(bottom=0, top=.95)

cbar_ax1 = fig.add_axes([0.48, 0.175, 0.015, 0.6])
cbar1 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm1, cmap=cmap1), orientation='vertical', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='bold', y=1.05, labelpad=-20, rotation=0)
cbar1.ax.tick_params(labelsize=14)
for tick in cbar1.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')

cbar_ax2 = fig.add_axes([0.905, 0.175, 0.015, 0.6])
cbar2 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm2, cmap=cmap2), orientation='vertical', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='bold', y=1.05, labelpad=-20, rotation=0)
cbar2.ax.tick_params(labelsize=14)
for tick in cbar2.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')


### LIG-PI Differences

In [ ]:
# calculate LIG--PI differences (from NAM_dD_figs notebook)

# dDp 
diff22p=12.42523674
diffdsdp=-0.67940596

In [ ]:
for key in ['diff']:
    dD[key] = dD['lig'] - dD['pi']
    d18O[key] = d18O['lig'] - d18O['pi']
    prec[key] = prec['lig'] - prec['pi']

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# var specs
im=5 #start month
em=9 #end month
# mean annual = 0:12
# JAS = 6:9
# ** for DJF, replace 'im:em' with '[11,0,1]'
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff=[12.42523674, -0.67940596]

# colormap
cmap1=plt.colormaps['PuOr_r']
vmin1=-2.5
vmax1=2.5
levels=np.linspace(vmin1, vmax1, 13)
norm1=mpl.colors.BoundaryNorm(levels, cmap1.N)

cmap2=plt.colormaps['BrBG']
vmin2=-3
vmax2=3
levels=np.linspace(vmin2, vmax2, 13)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)


# plot specs
lw=1
font={'color':  'k',
      'weight': 'bold',
      'size': 10,
      'horizontalalignment': 'left'}
titles=np.array(['$\Delta$$\delta$D$_{precip}$', '$\Delta$Precipitation'])

# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]


# ------------------- #
#      Make Plot      #
# ------------------- #
months=['January-February-March', 'June-July-August-September ']
t_months=months[1]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(11,5), subplot_kw={'projection': proj})
fig.text(.5,.825,'iCESM1.2 ' +t_months+ ' LIG$-$PI Differences', size=12, weight='bold', ha='center')

#dDp
dDdiff = dDweightedvalues_lig[im:em,:,:].mean(dim="month") - dDweightedvalues_pi[im:em,:,:].mean(dim="time")
cf1=ax[0].pcolormesh(lon, lat, dDdiff, cmap=cmap1, vmin=vmin1, vmax=vmax1, transform=trans)
ax[0].scatter(x=clons, y=clats, c=ddiff,
              cmap=cmap1, vmin=-5, vmax=5, alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
ax[0].text(-105.9, 23, u'12.4‰', fontsize=10, weight='bold', ha='left')
ax[0].text(-111, 28.35, u'-0.7‰', fontsize=10, weight='bold', ha='left')

# precip
pdiff = prec_lig[im:em,:,:].mean(dim="month") - prec_pi[im:em,:,:].mean(dim="time")
cf2=ax[1].pcolormesh(lon, lat, pdiff, cmap=cmap2, vmin=vmin2, vmax=vmax2, transform=trans)
ax[1].scatter(clons, clats, c='k', s=150, alpha=1, transform=trans, zorder=100)


for i in [0,1]:
    ax[i].coastlines()
    ax[i].add_feature(cfeature.BORDERS)
    ax[i].add_feature(cfeature.STATES, linewidth=0.5)
    ax[i].text(map_bnds[0], map_bnds[3]+0.5, titles[i], fontdict=font)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax[i].add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax[i].set_extent(map_bnds, crs=trans)
    if i==0:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.right_labels=False
    if i==1:
        gl=ax[i].gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
        gl.top_labels=False; gl.left_labels=False; gl.right_labels=False


fig.subplots_adjust(bottom=0, top=.95)

cbar_ax1 = fig.add_axes([0.4825, 0.1925, 0.015, 0.563])
cbar1 = fig.colorbar(cf, norm=norm1, orientation='vertical', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='bold', labelpad=10, rotation=270)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')

cbar_ax2 = fig.add_axes([0.905, 0.1925, 0.015, 0.563])
cbar2 = fig.colorbar(cf2, orientation='vertical', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='bold', labelpad=15, rotation=270)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')

#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf")

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# var specs
im=5 #start month
em=9 #end month
# mean annual = 0:12
# JAS = 6:9
# ** for DJF, replace 'im:em' with '[11,0,1]'
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff=[12.42523674, -0.67940596]

# colormap
cmap1=plt.colormaps['PuOr_r']
vmin1=-2.5
vmax1=2.5
levels=np.linspace(vmin1, vmax1, 13)
norm1=mpl.colors.BoundaryNorm(levels, cmap1.N)

cmap2=plt.colormaps['BrBG']
vmin2=-3
vmax2=3
levels=np.linspace(vmin2, vmax2, 13)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)


# plot specs
lw=1
font={'color':  'k',
      'weight': 'bold',
      'size': 10,
      'horizontalalignment': 'left'}
titles=np.array(['$\Delta$$\delta$D$_{precip}$', '$\Delta$Precipitation'])

# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]


# ------------------- #
#      Make Plot      #
# ------------------- #
months=['January-February-March', 'June-July-August-September ']
t_months=months[1]

fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(13,10), subplot_kw={'projection': proj}, layout='constrained')
fig.text(.5,1.05,'iCESM1.2 LIG$-$PI Differences', size=16, weight='bold', ha='center')

#dDp
cf1=ax[0,0].pcolormesh(lon, lat, dDdiff, cmap=cmap1, vmin=vmin1, vmax=vmax1, transform=trans)
ax[0,0].scatter(x=clons, y=clats, c=ddiff,
                cmap=cmap1, vmin=-5, vmax=5, alpha=1, edgecolor='k', s=150, transform=trans, zorder=100)
ax[0,0].text(-105.9, 23, u'12.4‰', fontsize=10, weight='bold', ha='left')
ax[0,0].text(-111, 28.35, u'-0.7‰', fontsize=10, weight='bold', ha='left')

# precip
cf2=ax[0,1].pcolormesh(lon, lat, pdiff, cmap=cmap2, vmin=vmin2, vmax=vmax2, transform=trans)
ax[0,1].scatter(clons, clats, c='k', s=150, alpha=1, transform=trans, zorder=100)


for i,ax in enumerate(ax.flat):
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS)
    ax.add_feature(cfeature.STATES, linewidth=0.5)
    #ax.text(map_bnds[0], map_bnds[3]+0.5, titles[i], fontdict=font)
    ring=LinearRing(list(zip([-113., -105, -105, -113.], [18,  18,  33,  33])))
    ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=1, linestyle='--', zorder=9)
    ax.set_extent(map_bnds, crs=trans)
    gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=False)
    #if i==1:
    #    gl=ax.gridlines(crs=trans, lw=0.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
    #    gl.top_labels=False; gl.left_labels=False; gl.right_labels=False


cbar_ax1 = fig.add_axes([0.0125, -0.05, 0.475, 0.025])
cbar1 = fig.colorbar(cf, norm=norm1, orientation='horizontal', cax=cbar_ax1)
cbar1.set_label(u'[‰]', weight='bold', labelpad=5, rotation=0)
cbar1.ax.tick_params(labelsize=10)
for tick in cbar1.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')

cbar_ax2 = fig.add_axes([0.515, -0.05, 0.475, 0.025])
cbar2 = fig.colorbar(cf2, orientation='horizontal', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='bold', labelpad=5, rotation=0)
cbar2.ax.tick_params(labelsize=10)
for tick in cbar2.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')

#plt.savefig("cesm1.2_LIG-PI_jas_dDp_precip.pdf")

In [ ]:
# -------------------- #
#       Settings       #
# -------------------- #
# var specs
im=6 #start month
em=9 #end month
# mean annual = 0:12
# JAS = 6:9
# ** for DJF, replace 'im:em' with '[11,0,1]'
clons=[-106.5183, -111.62] 
clats=[22.5183, 27.85]
ddiff=[12.42523674, -0.67940596]

# colormap
cmap2=plt.colormaps['PuOr']
vmin2=-1.5
vmax2=1.5
levels=np.linspace(vmin2, vmax2, 31)
norm2=mpl.colors.BoundaryNorm(levels, cmap2.N)


# plot specs
lw=1
font={'color':  'k',
      'weight': 'bold',
      'size': 14,
      'horizontalalignment': 'left'}
titles=np.array(['$\Delta$$\delta$D$_{precip}$', '$\Delta$Precipitation'])

# map specs
trans=ccrs.PlateCarree()
proj=ccrs.PlateCarree()
map_bnds=[-125., -85., 10., 42.]


# ------------------- #
#      Make Plot      #
# ------------------- #
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(10,7.5), subplot_kw={'projection': proj})
fig.text(.5,.825,'iCESM1.2 LIG$-$PI July-August-September Precipitation Difference', size=14, weight='bold', ha='center')

# precip
pdiff = prec_lig[im:em,:,:].mean(dim="month") - prec_pi[im:em,:,:].mean(dim="time")
ax.contourf(lon, lat,pdiff,cmap=cmap2,levels=levels,extend='both',transform=trans)
#ax.pcolormesh(lon, lat, pdiff, cmap=cmap2, vmin=vmin2, vmax=vmax2, transform=trans)
ax.scatter(clons, clats, c='k', s=250, alpha=1, transform=trans, zorder=100)

ax.coastlines()
ax.add_feature(cfeature.BORDERS)
ax.add_feature(cfeature.STATES, linewidth=0.5)
ring=LinearRing(list(zip([-112., -102, -102, -112.], [18,  18,  33,  33])))
ax.add_geometries([ring], crs=trans, fc='none', ec='k', lw=2, linestyle='--', zorder=9)
ax.set_extent(map_bnds, crs=trans)
gl=ax.gridlines(crs=trans, lw=.5, colors='black', alpha=1.0, linestyle='--', zorder=10, draw_labels=True)
gl.top_labels   = False; gl.right_labels = False
gl.xformatter = LONGITUDE_FORMATTER; gl.yformatter = LATITUDE_FORMATTER
gl.xlabel_style = {'color': 'black', 'weight': 'bold'}; gl.ylabel_style = {'color': 'black', 'weight': 'bold'}

fig.subplots_adjust(bottom=.1, top=.8)

cbar_ax2 = fig.add_axes([0.875, 0.175, 0.025, 0.5])
cbar2 = fig.colorbar(mpl.cm.ScalarMappable(norm=norm2, cmap=cmap2), orientation='vertical', cax=cbar_ax2)
cbar2.set_label('[mm day$^{-1}$]', weight='bold', y=1.05, labelpad=-35, rotation=0)
cbar2.ax.tick_params(labelsize=14)
for tick in cbar2.ax.yaxis.get_major_ticks():
    tick.label2.set_fontweight('bold')
    
#plt.savefig("cesm1.2_LIG-PI_jas_precip_diff.pdf")

## Temperature

In [ ]:
### +++ Load Data +++ ###

dpath0     = '/Users/dervlamk/My Drive/Research/iCESM/LIG127k/'

# Timeseries
filen      = 'TS.iLIG127k.nc'
dfile      = f'{dpath0}/timeseries/{filen}'
tsurf      = xr.open_dataset(f'{dfile}').TS-273.15
time       = xr.open_dataset(f'{dfile}').time
lat        = xr.open_dataset(f'{dfile}').lat
lon        = xr.open_dataset(f'{dfile}').lon

# Climatology
filen      = 'TS_monthly_climatology.iLIG127k.nc'
dfile      = f'{dpath0}/climatologies/{filen}'
tsurfmon   = xr.open_dataset(f'{dfile}').TS-273.15
tsurf_mean = tsurfmon.mean(dim="month")
tsurf_jfm  = tsurfmon[0:2,:,:].mean(dim="month")
tsurf_jja  = tsurfmon[6:8,:,:].mean(dim="month")

In [ ]:
### +++ Plot Mean TS +++ ###
trans    = ccrs.PlateCarree()
proj     = ccrs.PlateCarree()
fig, axs = plt.subplots(nrows=3, ncols=1, figsize=(24,22), subplot_kw={'projection': proj})

# colormap specs
cmap = plt.get_cmap('RdYlBu_r', 51)
vmin = -60.
vmax = 40.   

## PI Control ##
# t_surf climatology
cf0 = axs[0].pcolormesh(lon, lat,
                        tsurf_mean,
                        cmap=cmap,
                        vmin=vmin,
                        vmax=vmax,
                        transform=trans)
cbar = fig.colorbar(cf0,
                    shrink=0.8,
                    orientation='vertical',
                    ax=axs[0])
cbar.set_label('deg C',
               labelpad=-35,
               y=1.075,
               rotation=0)
# Subplot specs
axs[0].coastlines()
axs[0].set_global()
axs[0].set_title(("Mean Annual Tsurf"),
                 fontsize=14,
                 fontweight='bold')
gl = axs[0].gridlines(crs=proj,
                      linewidth=.5,
                      color='black',
                      alpha=0.5,
                      linestyle='--',
                      draw_labels=True)
gl.top_labels   = False
gl.right_labels = False
gl.xformatter   = LONGITUDE_FORMATTER
gl.yformatter   = LATITUDE_FORMATTER
gl.xlabel_style = {'color': 'black', 'weight': 'bold'}
gl.ylabel_style = {'color': 'black', 'weight': 'bold'}

# t_surf climatology
cf0 = axs[1].pcolormesh(lon, lat,
                        tsurf_jfm,
                        cmap=cmap,
                        vmin=vmin,
                        vmax=vmax,
                        transform=trans)
cbar = fig.colorbar(cf0,
                    shrink=0.8,
                    orientation='vertical',
                    ax=axs[1])
cbar.set_label('deg C',
               labelpad=-35,
               y=1.075,
               rotation=0)
# Subplot specs
axs[1].coastlines()
axs[1].set_global()
axs[1].set_title(("JFM Tsurf"),
                 fontsize=14,
                 fontweight='bold')
gl = axs[1].gridlines(crs=proj,
                      linewidth=.5,
                      color='black',
                      alpha=0.5,
                      linestyle='--',
                      draw_labels=True)
gl.top_labels   = False
gl.right_labels = False
gl.xformatter   = LONGITUDE_FORMATTER
gl.yformatter   = LATITUDE_FORMATTER
gl.xlabel_style = {'color': 'black', 'weight': 'bold'}
gl.ylabel_style = {'color': 'black', 'weight': 'bold'}


# t_surf climatology
cf2 = axs[2].pcolormesh(lon, lat,
                        tsurf_jja,
                        cmap=cmap,
                        vmin=vmin,
                        vmax=vmax,
                        transform=trans)
cbar = fig.colorbar(cf2,
                    shrink=0.8,
                    orientation='vertical',
                    ax=axs[2])
cbar.set_label('deg C',
               labelpad=-35,
               y=1.075,
               rotation=0)
# Subplot specs
axs[2].coastlines()
axs[2].set_global()
axs[2].set_title(("JAS Tsurf"),
                 fontsize=14,
                 fontweight='bold')
gl = axs[2].gridlines(crs=proj,
                      linewidth=.5,
                      color='black',
                      alpha=0.5,
                      linestyle='--',
                      draw_labels=True)
gl.top_labels   = False
gl.right_labels = False
gl.xformatter   = LONGITUDE_FORMATTER
gl.yformatter   = LATITUDE_FORMATTER
gl.xlabel_style = {'color': 'black', 'weight': 'bold'}
gl.ylabel_style = {'color': 'black', 'weight': 'bold'}

In [ ]:
### +++ Global Mean Surface Temperature timeseries +++ ###

#  Weight data based on latitude to account for differences in gridcell area
weights      = np.cos(np.deg2rad(lat))
weights.name = "weights" 
gmst_wtd     = tsurf.weighted(weights).mean(("lon", "lat"))

# Calculate annual mean from monthly data
gmstwtd_Ann  = gmst_wtd.rolling(time=12).mean()

# Make Fig
fig, axs = plt.subplots(nrows=1, ncols=1, figsize=(20,10))


# Monthly timeseries
axs.plot(time,
            gmst_wtd,
            color = 'red',
            linewidth=1,
            linestyle='-')
# Annual timeseries
axs.plot(time,
            gmstwtd_Ann,
            color = 'black',
            linewidth=2,
            linestyle='-')
# Subplot settings
axs.set_title(("Global Mean Surface Temperature"),
                 fontsize=14,
                 fontweight='bold')
axs.set(xlabel="Month",
           ylabel="Temperature (deg C)")
axs.legend(["Monthly", "Mean Annual"],
              loc=0,
              frameon=True,
              fontsize=12)